# prepare PATH

In [1]:
from pathlib import Path
import sys

repo_root = Path.cwd().resolve()
# step up the dir tree from this notebook to find the root dir of this project.
while not (repo_root / "src" / "__init__.py").is_file():
    parent = repo_root.parent
    if parent == repo_root:
        raise RuntimeError(
            "Could not find the repository root from the notebook's working directory."
        )
    repo_root = parent

sys.path.insert(0, str(repo_root))

In [2]:
# import system libraries
import torch
from copy import deepcopy

# import user-defined libraries
from src.prepared_data import load_entity

# import data processing libraries
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.compute as pc

# simple implementation flow
- [ ] load data into train, validation and test splits.
- [ ] windowize into train, validation and test windows. write a custom pytorch dataset and dataloader.
- [ ] write the training loop and tracking the best checkpoint using a metric.
- [ ] write the model.
- [ ] evaluate on test set, which means calculate `VUS-PR`, `VUS-ROC`, `Affiliation F1`, etc.

## load data

In [ ]:
# data = load_entity(
#     Path(
#         "../../data/preprocessed/swat/67e1e1753e66f44858d0b31ec38ccff60bb86e805b50a889d576fb02ca39e391/"
#     ),
#     "swat"
# )
# train = data["splits"]["train"]
# validation = data["splits"]["validation"]
# test = data["splits"]["validation"]
# labels = train["labels"]

In [ ]:
# print(len(train['x']))
# print(len(validation['x']))
# print(len(test['x']))

396000
99000
99000


In [ ]:
# # inspect keys of each split object
# data['splits']['train'].keys()

dict_keys(['x', 'labels', 'source_file', 'source_row', 'segments', 'identity'])

## dataset

In [7]:
train_table = pq.read_table(
    "../../data/preprocessed/swat/67e1e1753e66f44858d0b31ec38ccff60bb86e805b50a889d576fb02ca39e391/swat/train.parquet"
)
validation_table = pq.read_table(
    "../../data/preprocessed/swat/67e1e1753e66f44858d0b31ec38ccff60bb86e805b50a889d576fb02ca39e391/swat/validation.parquet"
)
test_table = pq.read_table(
    "../../data/preprocessed/swat/67e1e1753e66f44858d0b31ec38ccff60bb86e805b50a889d576fb02ca39e391/swat/test.parquet"
)

In [ ]:
# type(train_table.slice(offset=0, length=100))

In [ ]:
# arrays_dict = {
#     name: col.to_numpy() for name, col in zip(train_table.column_names,
#                                               train_table.columns)
# }
# arrays_dict

In [12]:
arr_2d = np.column_stack([col.to_numpy() for col in train_table.columns])
arr_2d.shape

(396000, 48)

In [ ]:
class SWaTDataset(torch.utils.data.Dataset):
    def __init__(self, split, stride, win_size, parquet_file, root_dir, transform=None):
        """
        Arguments:
            parquet_file (string): Path to the parquet file with labels.
            root_dir (string): Directory with all signals.
            transform (callable, optional): Optional transform to be applied on a sample.
        """
        self.split = split
        self.strid = stride
        self.win_size = win_size
        self.parquet_file = (
            parquet_file  # this file can be train, validation or test file.
        )
        self.root_dir = root_dir
        self.transform = transform

        # load series of signals
        self.table: pa.Table = pq.read_table(parquet_file)
        self.X = self.table.select(self.table.column_names[:-4])

        # if this is test split, then load series of labels
        if split == "test":
            self.Y = self.table.select(self.table.column_names[-2])

    def __len__(self):
        """
        Return the length of this dataset, aka the number of time-series windows.
        """
        return len(self.table) - self.win_size

    def __getitem__(self, idx):
        """
        Get the window starting at global index `idx` in `self.table`.
        """
        if idx > len(self.table) - self.win_size - 1:
            raise ValueError(
                "`idx` must not be after the starting point of the last window."
            )

        sample = None
        if self.split == "train":
            pa_slice = self.X.slice(offset=idx, length=self.win_size)
            np2darray = np.column_stack([col for col in pa_slice.columns])
            sample = {"win_values": np2darray}  # this is a window, which can be
            # multivariate or univariate.
        return sample

## mustard

In [23]:
x = torch.rand((4, 100, 48))
# map input vectors into embedding space.
d_model = 128
embd_layer = torch.nn.Linear(in_features=48, out_features=d_model)
relu = torch.nn.ReLU()
embeddings = relu(embd_layer(x))

# encode embeddings into latent vectors.
enc_1 = torch.nn.Conv1d(
    in_channels=100, out_channels=100, kernel_size=3, padding="same"
)
enc_2 = torch.nn.Conv1d(
    in_channels=100, out_channels=100, kernel_size=5, padding="same"
)
enc_3 = torch.nn.Conv1d(
    in_channels=100, out_channels=100, kernel_size=5, padding="same"
)

latent_vectors = enc_3(relu(enc_2(relu(enc_1(embeddings)))))
latent_vectors.shape

torch.Size([4, 100, 128])

In [24]:
# linear_weights = torch.ones(latent_vectors.shape[1])
entry_wise_weights = torch.ones((latent_vectors.shape[1], d_model))
pooled_vector = torch.ones(latent_vectors.shape[1]) @ (
    entry_wise_weights * latent_vectors
)
pooled_vector.shape

torch.Size([4, 128])

In [ ]:
class MustardNetwork(torch.nn.Module):
    def __init__(self, in_features, win_size, d_model: int = 128, grn_v3: bool = False):
        self.d_model = d_model
        self.embedding_layer = torch.nn.Linear(
            in_features=in_features, out_features=self.d_model
        )
        self.relu = torch.nn.ReLU()
        self.enc_layer1 = torch.nn.Conv1d(
            in_channels=win_size, out_channels=win_size, kernel_size=3, padding="same"
        )
        self.enc_layer2 = torch.nn.Conv1d(
            in_channels=win_size, out_channels=win_size, kernel_size=5, padding="same"
        )
        self.enc_layer3 = torch.nn.Conv1d(
            in_channels=win_size, out_channels=win_size, kernel_size=5, padding="same"
        )
        self.enc_layers = [self.enc_layer1, self.enc_layer2, self.enc_layer3]

        # use generalized residual network or not.
        self.grn_v3 = grn_v3
        if self.grn_v3 == True:
            self.input_dependent_weights = torch.zeros((self.d_model, 1))
            self.dim_dependent_weights = torch.ones(
                (self.d_model, len(self.enc_layers))
            )

    def forward(self, x):
        """
        x: [B, T, C] (torch.Tensor): input time-series window.
        """
        embeddings = self.embedding_layer(x)  # [B, T, 128]

        latent_vectors = embeddings
        if self.grn_v3 == False:
            for enc_layer in self.enc_layers:
                latent_vectors = enc_layer(latent_vectors) + latent_vectors
        elif self.grn_v3 == True:
            # https://arxiv.org/abs/2502.06785
            # https://share.gemini.google/Fhhv5x1k1L2i
            layerwise_outputs = [embeddings]
            for enc_layer in self.enc_layers:
                pass

## training and validation

## evaluation